# Fraud detection – candidate submissions
One run trains CatBoost, LightGBM and TabPFN once and writes **one submission CSV per candidate**, each in the sample format (`id,prediction`):

| File | Model(s) |
|---|---|
| `submission_tabpfn_catboost.csv` (also `submission.csv`) | TabPFN + CatBoost rank blend – recommended: the two strongest, most different models |
| `submission_catboost.csv` | CatBoost alone |
| `submission_lightgbm.csv` | LightGBM alone |
| `submission_tabpfn.csv` | TabPFN alone |

Upload each to the scoring site and keep the best. `prediction` is a fraud score in [0,1]: its ranking gives PR-AUC, and scores ≥ 0.5 are the fraud flags.

In [ ]:
%pip install -q tabpfn lightgbm catboost

import os, sys
import numpy as np
import pandas as pd

# SECURITY: never paste the token into the notebook. In Colab, add it under the key icon (Secrets) as TABPFN_TOKEN.
if "google.colab" in sys.modules:
    from google.colab import userdata
    os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

# Read from the repo's data/ folder if it exists; otherwise open an upload button (Colab)
def load_csv(name):
    for path in [f"../data/{name}", f"data/{name}", name]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    print(f"Upload: {name}")
    return pd.read_csv(list(files.upload().keys())[0])

df = load_csv("Track 2 Training Dataset.csv")
test_df = load_csv("Track 2 Testing Dataset.csv")
print("Train:", df.shape, " Test:", test_df.shape)

In [ ]:
target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]

# Adds per-row features (uses only that row's own values, so no label or cross-row leakage)
def engineer(d):
    d = d.copy()
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# --- DROP UNNECESSARY ID COLUMNS --- (id is a random label; checked: it carries no fraud signal)
X = engineer(df.drop(columns=["id", target_column]))
y = df[target_column].values
NUM_COLS = [c for c in X.columns if c not in CATS]

# Test set: same features, and exactly the same category lists as training
X_test = engineer(test_df.drop(columns=["id"]))
for c in CATS:
    X_test[c] = pd.Categorical(X_test[c].astype(str), categories=X[c].cat.categories)
X_test = X_test[X.columns]
print(f"{X.shape[1]} features, fraud rate {y.mean():.2%}")

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer
import lightgbm as lgb
from catboost import CatBoostClassifier
from tabpfn import TabPFNClassifier

def make_lgb():
    # Small, heavily regularised LightGBM (handles NaN and pandas categories natively).
    # Tuned by random search (70 configs, 5x2 CV) and confirmed on fresh splits: OOF PR-AUC 0.219 vs 0.207 before.
    # Tiny trees (4 leaves, >=150 rows each) can't memorise single frauds; unweighted (scale_pos_weight=1) was best.
    return lgb.LGBMClassifier(n_estimators=900, learning_rate=0.03, num_leaves=4, min_child_samples=150,
                              colsample_bytree=0.85, reg_lambda=10, reg_alpha=0.5, min_split_gain=0.01,
                              cat_smooth=100, min_data_per_group=50, random_state=0, verbose=-1)

def make_catboost():
    # Strongest single model in CV (ordered target statistics for categoricals, small trees, strong L2).
    # CatBoost needs categoricals as strings.
    to_str = FunctionTransformer(lambda d: d.astype({c: str for c in CATS}))
    return make_pipeline(to_str, CatBoostClassifier(iterations=800, depth=4, learning_rate=0.03, l2_leaf_reg=10,
                                                    cat_features=CATS, random_seed=0, verbose=0,
                                                    allow_writing_files=False))

def make_tabpfn():
    return TabPFNClassifier(ignore_pretraining_limits=True)

MODELS = {"catboost": make_catboost, "lightgbm": make_lgb, "tabpfn": make_tabpfn}

# Each candidate = the models it blends (a single model is just itself)
CANDIDATES = {
    "tabpfn_catboost": ["tabpfn", "catboost"],
    "catboost": ["catboost"],
    "lightgbm": ["lightgbm"],
    "tabpfn": ["tabpfn"],
}

# Probability of the fraud class
def p_fraud(model, X_):
    return model.predict_proba(X_)[:, list(model.classes_).index(1)]

# Rank-average blend: ranks make the models' different probability scales comparable
def blend(scores):
    return sum(pd.Series(s).rank(pct=True).values for s in scores) / len(scores)

In [ ]:
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold

# 5-fold CV: every row is scored once by models that never saw it (out-of-fold)
oof = {name: np.zeros(len(y)) for name in MODELS}
for fold, (tr, te) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=0).split(X, y)):
    for name, make in MODELS.items():
        oof[name][te] = p_fraud(make().fit(X.iloc[tr], y[tr]), X.iloc[te])
    print(f"fold {fold} done")

# Threshold: PR-AUC does not depend on it (it only uses the ranking). It sets the 0/1 fraud flags scored by F1 / Recall.
# For every cut-off (share of rows flagged) build the confusion matrix; pick the share with the best (smoothed) F1.
def pick_share(score):
    rows = []
    for share in np.arange(0.002, 0.0601, 0.0005):
        flag = (score >= np.quantile(score, 1 - share)).astype(int)
        tn, fp, fn, tp = confusion_matrix(y, flag).ravel()
        rows.append({"share": share, "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "F1": 2 * tp / (2 * tp + fp + fn)})
    cm = pd.DataFrame(rows)
    cm["F1_smooth"] = cm["F1"].rolling(5, center=True, min_periods=1).mean()   # don't pick a one-off noisy spike
    return cm.loc[cm["F1_smooth"].idxmax()]

results, share = [], {}
for cand, members in CANDIDATES.items():
    score = blend([oof[m] for m in members])
    best = pick_share(score)
    share[cand] = best["share"]
    results.append({"candidate": cand, "PR-AUC": average_precision_score(y, score), "flag share": best["share"],
                    "precision": best["precision"], "recall": best["recall"], "F1": best["F1"]})
pd.DataFrame(results).set_index("candidate").round(4)

In [ ]:
# Maps scores to [0,1] without changing their order (PR-AUC unchanged) and puts the flag threshold at 0.5
def to_submission_score(score, thr):
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

# Train every model once on all 20,000 training rows and score the test set
test_scores = {name: p_fraud(make().fit(X, y), X_test) for name, make in MODELS.items()}

# One submission file per candidate
written = []
for cand, members in CANDIDATES.items():
    score = blend([test_scores[m] for m in members])
    prob = to_submission_score(score, np.quantile(score, 1 - share[cand]))
    sub = pd.DataFrame({"id": test_df["id"], "prediction": prob})
    assert len(sub) == len(test_df) and sub["id"].is_unique
    assert sub["prediction"].between(0, 1).all() and sub["prediction"].nunique() > 2
    sub.to_csv(f"submission_{cand}.csv", index=False)
    written.append(f"submission_{cand}.csv")
    if cand == "tabpfn_catboost":
        sub.to_csv("submission.csv", index=False)   # recommended candidate
    print(f"submission_{cand}.csv: {len(sub)} rows, flagged (>= 0.5): {(prob >= 0.5).sum()}")

# In Colab: download all files at once as a zip
if "google.colab" in sys.modules:
    import zipfile
    from google.colab import files
    with zipfile.ZipFile("submissions.zip", "w") as z:
        for f in written:
            z.write(f)
    files.download("submissions.zip")